# Pipeline Visual Review: Merge Dataset to Train Model

This notebook is a readable, notebook-shaped version of the important project files from `03_merge_dataset.py` to `04_train_model.py`.

It is designed for understanding and visualization, not for re-running the full pipeline like a command. The full Python files are included below as source-code sections, and the executable cells are only for safely reading already-saved artifacts such as the parquet dataset, feature names, PCA files, and model results.

Main idea:

1. `config.py` defines paths and parameters.
2. `03_merge_dataset.py` reads saved feature files and creates the final parquet.
3. `04_train_model.py` reads that parquet and trains/evaluates a model.
4. The visualization cells here let you inspect what was already saved.


## How to Use This Notebook

Run only the safe inspection/visualization cells unless you intentionally want to execute training code yourself.

The source-code sections are shown inside Markdown code blocks, so they are readable but will not run automatically. This avoids accidentally re-merging the dataset or re-training the model.


## Safe Imports for Exploration

This cell is safe. It only imports libraries and your `config.py` module. It does not call `config.make_dirs()`, does not merge data, and does not train a model.


In [ ]:
from pathlib import Path
import json
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid")
except Exception:
    sns = None

import config

pd.set_option("display.max_columns", 140)
pd.set_option("display.max_colwidth", 140)


def exists(path):
    return Path(path).exists()


def read_json_if_exists(path):
    path = Path(path)
    if not path.exists():
        return None
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def read_csv_if_exists(path):
    path = Path(path)
    return pd.read_csv(path) if path.exists() else None


## Config Parameters

These are the most important values from `config.py` in table form. This is easier to read than scrolling through the whole file.


In [ ]:
config_values = {
    "BASE": config.BASE,
    "DATASET_DIR": config.DATASET_DIR,
    "METADATA_DIR": config.METADATA_DIR,
    "VIDEO_DIR": config.VIDEO_DIR,
    "AUDIO_DIR": config.AUDIO_DIR,
    "FEATURE_DIR": config.FEATURE_DIR,
    "HOOK_EMB_DIR": config.HOOK_EMB_DIR,
    "FULL_EMB_DIR": config.FULL_EMB_DIR,
    "TEXT_EMB_DIR": config.TEXT_EMB_DIR,
    "AUDIO_FEAT_DIR": config.AUDIO_FEAT_DIR,
    "WHISPER_DIR": config.WHISPER_DIR,
    "LLM_DIR": config.LLM_DIR,
    "PCA_DIR": config.PCA_DIR,
    "MODEL_DIR": config.MODEL_DIR,
    "RESULTS_DIR": config.RESULTS_DIR,
    "ALL_METADATA_CSV": config.ALL_METADATA_CSV,
    "EMBEDDING_INDEX_CSV": config.EMBEDDING_INDEX_CSV,
    "TEXT_INDEX_CSV": config.TEXT_INDEX_CSV,
    "AUDIO_INDEX_CSV": config.AUDIO_INDEX_CSV,
    "WHISPER_INDEX_CSV": config.WHISPER_INDEX_CSV,
    "FINAL_DATASET_PATH": config.FINAL_DATASET_PATH,
    "HOOK_SECONDS": config.HOOK_SECONDS,
    "HOOK_MAX_FRAMES": config.HOOK_MAX_FRAMES,
    "FULL_MAX_FRAMES": config.FULL_MAX_FRAMES,
    "CLIP_MODEL_NAME": config.CLIP_MODEL_NAME,
    "TEXT_MODEL_NAME": config.TEXT_MODEL_NAME,
    "WHISPER_MODEL_SIZE": config.WHISPER_MODEL_SIZE,
    "PCA_HOOK_COMPONENTS": config.PCA_HOOK_COMPONENTS,
    "PCA_FULL_COMPONENTS": config.PCA_FULL_COMPONENTS,
    "PCA_TEXT_COMPONENTS": config.PCA_TEXT_COMPONENTS,
    "TARGET_COL": config.TARGET_COL,
    "LEAKY_COLS": config.LEAKY_COLS,
    "META_COLS": config.META_COLS,
    "CV_FOLDS": config.CV_FOLDS,
    "RANDOM_STATE": config.RANDOM_STATE,
    "TEST_SIZE": config.TEST_SIZE,
}

config_df = pd.DataFrame(
    [{"parameter": k, "value": repr(v), "path_exists": exists(v) if isinstance(v, str) and ("/" in v or "\\" in v) else None}
     for k, v in config_values.items()]
)
config_df


## Full `config.py` Source

Source file: `config.py`

Read-only source view. Do not run this as a pipeline step.

```python
"""
config.py — Central configuration for the Viral Agent pipeline.
Edit paths and hyperparameters here. All scripts import from this file.
"""

import os

BASE = os.environ.get("VIRAL_AGENT_BASE", "/content/drive/MyDrive/viral_agent")

DATASET_DIR  = f"{BASE}/datasets"
METADATA_DIR = f"{DATASET_DIR}/metadata"
VIDEO_DIR    = f"{DATASET_DIR}/videos"
AUDIO_DIR    = f"{DATASET_DIR}/audios"          # extracted .wav files

FEATURE_DIR      = f"{BASE}/features"
HOOK_EMB_DIR     = f"{FEATURE_DIR}/hook_embeddings"
FULL_EMB_DIR     = f"{FEATURE_DIR}/full_video_embeddings"
TEXT_EMB_DIR     = f"{FEATURE_DIR}/text_embeddings"
AUDIO_FEAT_DIR   = f"{FEATURE_DIR}/audio_features"
WHISPER_DIR      = f"{FEATURE_DIR}/whisper_transcripts"
LLM_DIR          = f"{FEATURE_DIR}/llm_features"
PCA_DIR          = f"{FEATURE_DIR}/pca_reduced"

MODEL_DIR        = f"{BASE}/models"
RESULTS_DIR      = f"{BASE}/results"

ALL_METADATA_CSV     = f"{METADATA_DIR}/all_metadata.csv"
EMBEDDING_INDEX_CSV  = f"{FEATURE_DIR}/embedding_index.csv"
TEXT_INDEX_CSV       = f"{FEATURE_DIR}/text_embedding_index.csv"
AUDIO_INDEX_CSV      = f"{FEATURE_DIR}/audio_feature_index.csv"
WHISPER_INDEX_CSV    = f"{FEATURE_DIR}/whisper_transcript_index.csv"
FINAL_DATASET_PATH   = f"{FEATURE_DIR}/final_model_dataset.parquet"

# Feature Extraction 
HOOK_SECONDS  = 3          # seconds to consider as "hook"
HOOK_MAX_FRAMES = 12       # frames sampled from hook
FULL_MAX_FRAMES = 24       # frames sampled from full video
CLIP_MODEL_NAME = "openai/clip-vit-base-patch32"
CLIP_DIM = 512

TEXT_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
TEXT_DIM = 384

WHISPER_MODEL_SIZE = "base"   # tiny | base | small | medium | large

# PCA 
PCA_HOOK_COMPONENTS  = 64
PCA_FULL_COMPONENTS  = 64
PCA_TEXT_COMPONENTS  = 48

# LLM (Groq) 
GROQ_MODEL = "llama-3.3-70b-versatile"
GROQ_RATE_LIMIT_SLEEP = 0.5   # seconds between API calls

# Training 
TARGET_COL   = "log_views"
# Columns that MUST be excluded (post-publication data leakage)
LEAKY_COLS   = ["likes", "comments", "views", "log_views", "video_id", "platform", "shares"]
# NEW (Safe)
META_COLS    = ["video_id", "platform", "duration_seconds", "log_views"]
# META_COLS    = ["video_id", "platform", "views", "likes", "comments", "duration_seconds", "log_views"]

CV_FOLDS     = 5
RANDOM_STATE = 42
TEST_SIZE    = 0.2

# XGBoost defaults
XGB_PARAMS = {
    "n_estimators": 300,
    "max_depth": 4,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "min_child_weight": 3,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
}

# LightGBM defaults
LGBM_PARAMS = {
    "n_estimators": 300,
    "max_depth": 4,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "min_child_samples": 5,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "verbose": -1,
}

# MLP defaults
MLP_HIDDEN_DIMS  = [512, 256, 128]
MLP_DROPOUT      = 0.3
MLP_EPOCHS       = 100
MLP_BATCH_SIZE   = 32
MLP_LR           = 1e-3

# Helpers 
def make_dirs():
    """Create all required directories."""
    dirs = [
        METADATA_DIR, VIDEO_DIR, AUDIO_DIR,
        HOOK_EMB_DIR, FULL_EMB_DIR, TEXT_EMB_DIR,
        AUDIO_FEAT_DIR, WHISPER_DIR, LLM_DIR, PCA_DIR,
        MODEL_DIR, RESULTS_DIR,
    ]
    for d in dirs:
        os.makedirs(d, exist_ok=True)
    print("All directories ready.")

```


# Part 1: Merge Dataset Logic

This is the source of `03_merge_dataset.py`. Its job is to read the already extracted features, combine them into one table, apply PCA, create the target `log_views`, and save the final parquet.

In this notebook, the source is displayed for study only. The executable cells after it read the already-saved output.


## Full `03_merge_dataset.py` Source

Source file: `03_merge_dataset.py`

Important sections to notice: `load_and_merge_indexes`, `_load_npy_stack`, `fit_and_apply_pca`, `load_acoustic_features`, `load_llm_features`, and `main`.

```python
"""
03_merge_dataset.py
────────────────────
Loads all features extracted by 02_feature_extraction.py, merges them
with metadata, applies PCA dimensionality reduction, and saves a single
Parquet file ready for model training.

Steps:
  1. Load metadata + all index CSVs
  2. Inner-join to keep only videos with ALL features
  3. Stack numpy arrays → dense feature matrix
  4. Fit PCA on embeddings (hook / full / text), save PCA objects
  5. Add acoustic + LLM features
  6. Save final_dataset.parquet + feature_names.json

Usage:
    python 03_merge_dataset.py
    python 03_merge_dataset.py --no-pca          # skip PCA, keep raw dims
    python 03_merge_dataset.py --pca-components 32 32 24

Requirements:
    pip install pandas numpy scikit-learn pyarrow
"""

import argparse
import json
import logging
import os
import pickle
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm

sys.path.insert(0, str(Path(__file__).parent))
import config

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


# ────────────────────────────────────────────────────────────────────────────
# 1. Load indexes and merge


def load_and_merge_indexes(df: pd.DataFrame) -> pd.DataFrame:
    """Inner-join metadata with all feature indexes so we keep only complete rows."""
    required = {
        "clip":     config.EMBEDDING_INDEX_CSV,
        "text":     config.TEXT_INDEX_CSV,
        "whisper":  config.WHISPER_INDEX_CSV,
        "acoustic": config.AUDIO_INDEX_CSV,
    }

    merged = df.copy()
    for name, path in required.items():
        if not os.path.isfile(path):
            log.warning("Index not found, skipping join: %s", path)
            continue
        idx = pd.read_csv(path)
        idx["video_id"] = idx["video_id"].astype(str)
        idx = idx.drop_duplicates(subset=["video_id"], keep="first")

        before = len(merged)
        merged = merged.merge(idx, on="video_id", how="inner")
        log.info("  After joining %s: %d rows", name, len(merged))
    return merged.reset_index(drop=True)


def _load_npy_stack(paths: pd.Series, label: str) -> tuple[np.ndarray, list[int]]:
    """Load numpy files and stack into matrix. Returns (matrix, valid_indices)."""
    vectors, valid = [], []
    for i, p in enumerate(paths):
        try:
            v = np.load(str(p))
            vectors.append(v)
            valid.append(i)
        except Exception as e:
            log.debug("Cannot load %s: %s", p, e)
    if not vectors:
        raise RuntimeError(f"No valid numpy files for {label}")
    mat = np.vstack(vectors)
    log.info("  %-6s embeddings: shape=%s", label, mat.shape)
    return mat.astype(np.float32), valid



def fit_and_apply_pca(X: np.ndarray, n_components: int, label: str, save: bool = True) -> tuple[np.ndarray, PCA]:
    """Fit PCA and return reduced matrix + fitted PCA object."""
    n_components = min(n_components, X.shape[0] - 1, X.shape[1])
    log.info("  PCA %s: %d → %d dims", label, X.shape[1], n_components)
    pca = PCA(n_components=n_components, random_state=config.RANDOM_STATE)
    X_reduced = pca.fit_transform(X)

    evr = pca.explained_variance_ratio_.sum()
    log.info("    Explained variance: %.1f%%", evr * 100)

    if save:
        pca_path = os.path.join(config.PCA_DIR, f"{label}_pca.pkl")
        with open(pca_path, "wb") as f:
            pickle.dump(pca, f)
        log.info("    PCA saved → %s", pca_path)

    return X_reduced.astype(np.float32), pca



# 4. Acoustic features

def load_acoustic_features(df: pd.DataFrame) -> pd.DataFrame:
    """Load per-video acoustic JSON files and return a flat DataFrame."""
    rows = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Acoustic features"):
        vid = str(row["video_id"])
        p = row.get("audio_feat_path", "")
        if pd.notna(p) and os.path.isfile(str(p)):
            try:
                with open(p, "r") as f:
                    feats = json.load(f)
                feats["video_id"] = vid
                rows.append(feats)
            except Exception:
                rows.append({"video_id": vid})
        else:
            rows.append({"video_id": vid})

    acdf = pd.DataFrame(rows).fillna(0.0)
    acdf["video_id"] = acdf["video_id"].astype(str)
    log.info("Acoustic feature columns: %d", acdf.shape[1] - 1)
    return acdf


# 5. LLM features


def load_llm_features(df: pd.DataFrame) -> pd.DataFrame:
    """Load LLM JSON files, encode categoricals, return feature DataFrame."""
    rows = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc="LLM features"):
        vid = str(row["video_id"])
        p = os.path.join(config.LLM_DIR, f"{vid}_llm.json")
        if os.path.isfile(p):
            try:
                with open(p, "r") as f:
                    d = json.load(f)
                rows.append(d)
            except Exception:
                rows.append({"video_id": vid})
        else:
            rows.append({"video_id": vid})

    llm_df = pd.DataFrame(rows)
    if llm_df.empty:
        log.warning("No LLM features found — skipping")
        return pd.DataFrame({"video_id": df["video_id"].astype(str)})

    llm_df["video_id"] = llm_df["video_id"].astype(str)

    # Numeric scores 
    for col in ["hook_score", "clarity_score", "quality_score"]:
        if col in llm_df.columns:
            llm_df[col] = pd.to_numeric(llm_df[col], errors="coerce").fillna(0)

    # List features → counts 
        if col in llm_df.columns:
            llm_df[f"num_{col}"] = llm_df[col].apply(
                lambda x: len(x) if isinstance(x, list) else 0
            )

    # Categorical → one-hot 
    cat_cols = [c for c in ["hook_type", "tone", "emotion", "content_category"] if c in llm_df.columns]
    keep_cols = (
        ["video_id"]
        + [c for c in ["hook_score", "clarity_score", "quality_score"] if c in llm_df.columns]
        + [f"num_{c}" for c in ["strengths", "weaknesses", "engagement_triggers"] if f"num_{c}" in llm_df.columns]
        + cat_cols
    )
    llm_df = llm_df[[c for c in keep_cols if c in llm_df.columns]]
    llm_df = pd.get_dummies(llm_df, columns=cat_cols)

    log.info("LLM feature columns: %d", llm_df.shape[1] - 1)
    return llm_df


# ─────────────────────────────────────────────────────────────────────────────
# 6. Cosine similarity feature
# ─────────────────────────────────────────────────────────────────────────────

def hook_full_cosine(hook_mat: np.ndarray, full_mat: np.ndarray) -> np.ndarray:
    """Scalar cosine similarity between hook and full embedding per video."""
    h_norm = hook_mat / (np.linalg.norm(hook_mat, axis=1, keepdims=True) + 1e-8)
    f_norm = full_mat / (np.linalg.norm(full_mat, axis=1, keepdims=True) + 1e-8)
    sim = (h_norm * f_norm).sum(axis=1, keepdims=True)
    return sim.astype(np.float32)


# ─────────────────────────────────────────────────────────────────────────────
# Main
# ─────────────────────────────────────────────────────────────────────────────

def parse_args():
    p = argparse.ArgumentParser(description="Merge features and build final dataset")
    p.add_argument("--no-pca", action="store_true", help="Skip PCA, keep raw embedding dims")
    p.add_argument(
        "--pca-components",
        nargs=3,
        type=int,
        default=[config.PCA_HOOK_COMPONENTS, config.PCA_FULL_COMPONENTS, config.PCA_TEXT_COMPONENTS],
        metavar=("HOOK", "FULL", "TEXT"),
        help="Number of PCA components for hook / full / text embeddings",
    )
    return p.parse_args()


def main():
    args = parse_args()
    config.make_dirs()

    # ── Load metadata ──────────────────────────────────────────────────────
    if not os.path.isfile(config.ALL_METADATA_CSV):
        log.error("Metadata CSV not found: %s", config.ALL_METADATA_CSV)
        sys.exit(1)

    df = pd.read_csv(config.ALL_METADATA_CSV)
    df["video_id"] = df["video_id"].astype(str)
    log.info("Metadata: %d rows", len(df))

    # ── Merge indexes ──────────────────────────────────────────────────────
    log.info("Merging feature indexes…")
    df = load_and_merge_indexes(df)
    if len(df) == 0:
        log.error("No videos survive all inner joins. Check that feature extraction ran successfully.")
        sys.exit(1)

    # ── Load embedding matrices ────────────────────────────────────────────
    log.info("Loading embedding arrays…")
    hook_mat, hook_valid = _load_npy_stack(df["hook_path"], "hook")
    full_mat, full_valid = _load_npy_stack(df["full_path"], "full")
    text_mat, text_valid = _load_npy_stack(df["text_path"], "text")

    # Keep only rows that are valid in ALL three
    valid_rows = sorted(set(hook_valid) & set(full_valid) & set(text_valid))
    if len(valid_rows) < len(df):
        log.warning("Dropping %d rows due to missing numpy files", len(df) - len(valid_rows))
    df = df.iloc[valid_rows].reset_index(drop=True)
    hook_mat = hook_mat[[hook_valid.index(i) for i in valid_rows]]
    full_mat = full_mat[[full_valid.index(i) for i in valid_rows]]
    text_mat = text_mat[[text_valid.index(i) for i in valid_rows]]

    log.info("Valid rows after loading: %d", len(df))

    # ── PCA ────────────────────────────────────────────────────────────────
    if not args.no_pca:
        log.info("Applying PCA…")
        h_comp, f_comp, t_comp = args.pca_components
        hook_mat, _ = fit_and_apply_pca(hook_mat, h_comp, "hook")
        full_mat, _ = fit_and_apply_pca(full_mat, f_comp, "full")
        text_mat, _ = fit_and_apply_pca(text_mat, t_comp, "text")
    else:
        log.info("PCA skipped (--no-pca flag).")

    # ── Build DataFrame columns ────────────────────────────────────────────
    hook_cols = [f"hook_{i}" for i in range(hook_mat.shape[1])]
    full_cols = [f"full_{i}" for i in range(full_mat.shape[1])]
    text_cols = [f"text_{i}" for i in range(text_mat.shape[1])]

    hook_df = pd.DataFrame(hook_mat, columns=hook_cols)
    full_df = pd.DataFrame(full_mat, columns=full_cols)
    text_df = pd.DataFrame(text_mat, columns=text_cols)

    # ── Hook-Full cosine similarity ────────────────────────────────────────
    # Use raw (before PCA) embeddings for better geometric accuracy
    sim = hook_full_cosine(hook_mat, full_mat)
    sim_df = pd.DataFrame(sim, columns=["hook_full_cosine_sim"])

    # ── Target variable ────────────────────────────────────────────────────
    df["log_views"] = np.log1p(df["views"])

    # ── Acoustic features ──────────────────────────────────────────────────
    acoustic_df = pd.DataFrame()
    if os.path.isfile(config.AUDIO_INDEX_CSV):
        acoustic_df = load_acoustic_features(df)
        acoustic_df = acoustic_df.set_index("video_id").reindex(df["video_id"]).reset_index()
        # Remove duration_s if already in metadata
        acoustic_df = acoustic_df.drop(columns=["duration_s"], errors="ignore")

    # ── LLM features ──────────────────────────────────────────────────────
    llm_df = load_llm_features(df)
    llm_df = llm_df.set_index("video_id").reindex(df["video_id"]).reset_index()
    llm_df = llm_df.fillna(0)

    # ── Assemble final dataset ─────────────────────────────────────────────
    log.info("Assembling final dataset…")
    meta_cols = [c for c in config.META_COLS if c in df.columns]

    parts = [
        df[meta_cols].reset_index(drop=True),
        hook_df,
        full_df,
        text_df,
        sim_df,
    ]
    if not acoustic_df.empty:
        acoustic_df = acoustic_df.drop(columns=["video_id"], errors="ignore").reset_index(drop=True)
        parts.append(acoustic_df)
    if not llm_df.empty:
        llm_df = llm_df.drop(columns=["video_id"], errors="ignore").reset_index(drop=True)
        parts.append(llm_df)

    final_df = pd.concat(parts, axis=1)
    # drop any boolean columns left from pd.get_dummies (convert to int)
    bool_cols = final_df.select_dtypes(include="bool").columns
    final_df[bool_cols] = final_df[bool_cols].astype(int)

    final_df = final_df.fillna(0)
    log.info("Final dataset: %s", final_df.shape)

    # ── Save ───────────────────────────────────────────────────────────────
    final_df.to_parquet(config.FINAL_DATASET_PATH, index=False)
    log.info("Saved → %s", config.FINAL_DATASET_PATH)

    # Save feature names for inference scripts
    feature_cols = [c for c in final_df.columns if c not in config.LEAKY_COLS + ["video_id", "platform"]]
    names_path = os.path.join(config.FEATURE_DIR, "feature_names.json")
    with open(names_path, "w") as f:
        json.dump(feature_cols, f, indent=2)
    log.info("Feature names → %s  (%d features)", names_path, len(feature_cols))

    print("\n📊 Dataset summary:")
    print(final_df[["platform", "duration_seconds", "log_views"]].describe())


if __name__ == "__main__":
    main()

```


## What `03_merge_dataset.py` Produces

Expected outputs from the merge step:

- `features/final_model_dataset.parquet`
- `features/feature_names.json`
- `features/pca_reduced/hook_pca.pkl`
- `features/pca_reduced/full_pca.pkl`
- `features/pca_reduced/text_pca.pkl`

The next cells inspect those files if they already exist.


In [ ]:
merge_outputs = {
    "final parquet": config.FINAL_DATASET_PATH,
    "feature names": str(Path(config.FEATURE_DIR) / "feature_names.json"),
    "hook PCA": str(Path(config.PCA_DIR) / "hook_pca.pkl"),
    "full PCA": str(Path(config.PCA_DIR) / "full_pca.pkl"),
    "text PCA": str(Path(config.PCA_DIR) / "text_pca.pkl"),
}

pd.DataFrame([
    {"artifact": name, "path": path, "exists": exists(path)}
    for name, path in merge_outputs.items()
])


## Load Final Parquet as a DataFrame

This is the key reuse step. A parquet file becomes a normal pandas DataFrame after `pd.read_parquet(...)`.


In [ ]:
if exists(config.FINAL_DATASET_PATH):
    df = pd.read_parquet(config.FINAL_DATASET_PATH)
    print("Loaded final dataset:", df.shape)
else:
    df = pd.DataFrame()
    print("Final parquet not found:", config.FINAL_DATASET_PATH)

df.head()


## Dataset Shape and Column Types

In [ ]:
if not df.empty:
    display(df.dtypes.value_counts().rename("count").to_frame())
    display(pd.DataFrame({
        "column": df.columns,
        "dtype": [str(t) for t in df.dtypes],
        "missing": [int(df[c].isna().sum()) for c in df.columns],
        "missing_pct": [float(df[c].isna().mean() * 100) for c in df.columns],
    }).head(80))
else:
    print("No dataset loaded yet.")


## Feature Family Overview

This shows how the final table is composed: visual PCA features, text PCA features, LLM features, acoustic/other numeric features, and safe metadata.


In [ ]:
if not df.empty:
    feature_names_path = Path(config.FEATURE_DIR) / "feature_names.json"
    feature_cols = read_json_if_exists(feature_names_path)
    if feature_cols is None:
        feature_cols = [c for c in df.columns if c not in config.LEAKY_COLS + ["video_id", "platform"]]
    feature_cols = [c for c in feature_cols if c in df.columns]

    families = {
        "hook visual PCA": [c for c in feature_cols if c.startswith("hook_")],
        "full video visual PCA": [c for c in feature_cols if c.startswith("full_")],
        "text PCA": [c for c in feature_cols if c.startswith("text_")],
        "LLM numeric": [c for c in feature_cols if c in ["hook_score", "clarity_score", "quality_score"] or c.startswith("num_")],
        "LLM one-hot categories": [c for c in feature_cols if c.startswith(("hook_type_", "tone_", "emotion_", "content_category_"))],
        "hook/full similarity": [c for c in feature_cols if c == "hook_full_cosine_sim"],
        "safe metadata": [c for c in feature_cols if c in ["duration_seconds"]],
    }
    counted = set(sum(families.values(), []))
    families["other/acoustic"] = [c for c in feature_cols if c not in counted]

    family_df = pd.DataFrame({"family": list(families), "n_columns": [len(v) for v in families.values()]})
    display(family_df)

    plt.figure(figsize=(9, 4))
    plt.barh(family_df["family"], family_df["n_columns"], color="#2f5d7c")
    plt.xlabel("Number of columns")
    plt.title("Feature Families in Final Parquet")
    plt.tight_layout()
    plt.show()
else:
    print("No dataset loaded yet.")


## Target Distribution

The training target is `log_views`, not raw `views`. This usually makes viral-count prediction less dominated by huge outliers.


In [ ]:
if not df.empty and "log_views" in df.columns:
    ncols = 2 if "views" in df.columns else 1
    fig, axes = plt.subplots(1, ncols, figsize=(12, 4))
    if ncols == 1:
        axes = [axes]

    if "views" in df.columns:
        axes[0].hist(df["views"].dropna(), bins=40, color="#a23e48")
        axes[0].set_title("Raw views")
        axes[0].set_xlabel("views")
        target_ax = axes[1]
    else:
        target_ax = axes[0]

    target_ax.hist(df["log_views"].dropna(), bins=40, color="#397367")
    target_ax.set_title("Target: log_views")
    target_ax.set_xlabel("log_views")
    plt.tight_layout()
    plt.show()
else:
    print("Need a loaded dataset with log_views.")


## Platform, Duration, and Target Visualizations

In [ ]:
if not df.empty:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    if "platform" in df.columns:
        df["platform"].value_counts().plot(kind="bar", ax=axes[0], color="#2f5d7c")
        axes[0].set_title("Videos by platform")
    else:
        axes[0].text(0.5, 0.5, "No platform column", ha="center", va="center")

    if "duration_seconds" in df.columns:
        axes[1].hist(df["duration_seconds"].dropna(), bins=30, color="#d08c60")
        axes[1].set_title("Duration distribution")
        axes[1].set_xlabel("seconds")
    else:
        axes[1].text(0.5, 0.5, "No duration_seconds column", ha="center", va="center")

    if "duration_seconds" in df.columns and "log_views" in df.columns:
        axes[2].scatter(df["duration_seconds"], df["log_views"], alpha=0.45, color="#397367")
        axes[2].set_title("Duration vs log_views")
        axes[2].set_xlabel("duration_seconds")
        axes[2].set_ylabel("log_views")
    else:
        axes[2].text(0.5, 0.5, "Need duration + log_views", ha="center", va="center")

    plt.tight_layout()
    plt.show()
else:
    print("No dataset loaded yet.")


## PCA Objects Saved by Merge Step

This reads the PCA `.pkl` files that `03_merge_dataset.py` saved. It does not fit PCA again.


In [ ]:
pca_rows = []
for label in ["hook", "full", "text"]:
    path = Path(config.PCA_DIR) / f"{label}_pca.pkl"
    row = {"label": label, "path": str(path), "exists": path.exists()}
    if path.exists():
        with open(path, "rb") as f:
            pca = pickle.load(f)
        row.update({
            "input_dimensions": getattr(pca, "n_features_in_", None),
            "n_components": getattr(pca, "n_components_", None),
            "explained_variance_total": float(np.sum(getattr(pca, "explained_variance_ratio_", []))),
        })
    pca_rows.append(row)

pca_df = pd.DataFrame(pca_rows)
display(pca_df)

plot_df = pca_df.dropna(subset=["explained_variance_total"])
if len(plot_df):
    plt.figure(figsize=(7, 4))
    plt.bar(plot_df["label"], plot_df["explained_variance_total"], color="#397367")
    plt.ylim(0, 1)
    plt.ylabel("Total explained variance ratio")
    plt.title("Saved PCA Compression Summary")
    plt.tight_layout()
    plt.show()


## Correlations With `log_views`

This is just exploratory. Correlation is not feature importance, but it is useful for orientation.


In [ ]:
if not df.empty and "log_views" in df.columns:
    numeric = df.select_dtypes(include=[np.number])
    corr = numeric.corr(numeric_only=True)["log_views"].drop("log_views", errors="ignore").dropna()
    top_corr = corr.reindex(corr.abs().sort_values(ascending=False).head(25).index)
    display(top_corr.rename("corr_with_log_views").to_frame())

    plt.figure(figsize=(8, 8))
    colors = ["#397367" if v > 0 else "#a23e48" for v in top_corr.values]
    plt.barh(top_corr.index[::-1], top_corr.values[::-1], color=colors[::-1])
    plt.xlabel("Correlation")
    plt.title("Top Numeric Correlations With log_views")
    plt.tight_layout()
    plt.show()
else:
    print("Need a loaded dataset with log_views.")


# Part 2: Train Model Logic

This is the source of `04_train_model.py`. Its job is to read the parquet made by `03_merge_dataset.py`, build `X` and `y`, train a selected model, evaluate it, and save model artifacts.

Again: shown as source for reading, not executed automatically.


## Full `04_train_model.py` Source

Source file: `04_train_model.py`

Important sections to notice: `load_dataset`, model builders, `cross_validate`, hold-out evaluation, model saving, and feature importance saving.

```python
"""
04_train_model.py
──────────────────
Trains a viral-video regression model on the merged feature dataset
produced by 03_merge_dataset.py.

Supported model architectures
  --model xgb   → XGBoost  (default, best for tabular + PCA features)
  --model lgbm  → LightGBM (faster, similar accuracy)
  --model mlp   → PyTorch MLP (jointly learns from all modalities)
  --model rf    → Random Forest (legacy baseline)

Outputs
  models/best_model.pkl         (XGB / LGBM / RF)
  models/best_model_mlp.pt      (MLP state-dict)
  results/cv_scores.json        cross-validation metrics
  results/feature_importance.csv (XGB / LGBM / RF only)

Usage
  python 04_train_model.py
  python 04_train_model.py --model lgbm
  python 04_train_model.py --model mlp --epochs 150
  python 04_train_model.py --model xgb --tune     # Optuna HPO (optional)

Requirements
  pip install xgboost lightgbm scikit-learn optuna torch pandas pyarrow
"""

from __future__ import annotations

import argparse
import json
import logging
import os
import pickle
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.model_selection import KFold, train_test_split
from sklearn.preprocessing import StandardScaler

sys.path.insert(0, str(Path(__file__).parent))
import config

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


# ═════════════════════════════════════════════════════════════════════════════
# Helpers
# ═════════════════════════════════════════════════════════════════════════════

def load_dataset() -> tuple[np.ndarray, np.ndarray, list[str]]:
    """Load final_dataset.parquet → X, y arrays and feature names."""
    if not os.path.isfile(config.FINAL_DATASET_PATH):
        log.error("Dataset not found: %s\nRun 03_merge_dataset.py first.", config.FINAL_DATASET_PATH)
        sys.exit(1)

    df = pd.read_parquet(config.FINAL_DATASET_PATH)
    log.info("Loaded dataset: %s", df.shape)

    # Load pre-saved feature names (respects leaky-col exclusion from step 03)
    names_path = os.path.join(config.FEATURE_DIR, "feature_names.json")
    if os.path.isfile(names_path):
        with open(names_path) as f:
            feature_cols = json.load(f)
        # Keep only columns that actually exist in this parquet
        feature_cols = [c for c in feature_cols if c in df.columns]
    else:
        # Fallback: exclude known leaky and meta columns
        feature_cols = [
            c for c in df.columns
            if c not in config.LEAKY_COLS + ["video_id", "platform",
                                              "title", "url", "uploader",
                                              "upload_date", "local_video_path",
                                              "hook_path", "full_path",
                                              "text_path", "audio_feat_path",
                                              "transcript_path"]
        ]

    if config.TARGET_COL not in df.columns:
        log.error("Target column '%s' not found in dataset.", config.TARGET_COL)
        sys.exit(1)

    X = df[feature_cols].values.astype(np.float32)
    y = df[config.TARGET_COL].values.astype(np.float32)

    log.info("Features: %d  |  Samples: %d", X.shape[1], X.shape[0])
    log.info("Target   mean=%.3f  std=%.3f  min=%.3f  max=%.3f",
             y.mean(), y.std(), y.min(), y.max())

    return X, y, feature_cols


def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae  = float(mean_absolute_error(y_true, y_pred))
    r2   = float(r2_score(y_true, y_pred))
    return {"rmse": rmse, "mae": mae, "r2": r2}


def cross_validate(model_factory, X: np.ndarray, y: np.ndarray,
                   n_folds: int = config.CV_FOLDS) -> list[dict]:
    """k-fold CV returning per-fold metrics."""
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=config.RANDOM_STATE)
    fold_metrics = []

    for fold, (tr_idx, va_idx) in enumerate(kf.split(X), 1):
        X_tr, X_va = X[tr_idx], X[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]

        model = model_factory()
        model.fit(X_tr, y_tr)
        preds = model.predict(X_va)

        m = regression_metrics(y_va, preds)
        m["fold"] = fold
        fold_metrics.append(m)
        log.info("  Fold %d/%d  RMSE=%.4f  MAE=%.4f  R²=%.4f",
                 fold, n_folds, m["rmse"], m["mae"], m["r2"])

    return fold_metrics


def summarise_cv(fold_metrics: list[dict]) -> dict:
    summary = {}
    for key in ["rmse", "mae", "r2"]:
        vals = [m[key] for m in fold_metrics]
        summary[key] = {"mean": float(np.mean(vals)), "std": float(np.std(vals))}
    return summary


# ═════════════════════════════════════════════════════════════════════════════
# Model builders
# ═════════════════════════════════════════════════════════════════════════════

def build_xgb(params: dict | None = None):
    try:
        from xgboost import XGBRegressor
    except ImportError:
        log.error("xgboost not installed: pip install xgboost")
        sys.exit(1)
    p = {**config.XGB_PARAMS, **(params or {})}
    return XGBRegressor(**p)


def build_lgbm(params: dict | None = None):
    try:
        from lightgbm import LGBMRegressor
    except ImportError:
        log.error("lightgbm not installed: pip install lightgbm")
        sys.exit(1)
    p = {**config.LGBM_PARAMS, **(params or {})}
    return LGBMRegressor(**p)


def build_rf(params: dict | None = None):
    from sklearn.ensemble import RandomForestRegressor
    defaults = {
        "n_estimators": 300,
        "max_depth": None,
        "min_samples_leaf": 2,
        "n_jobs": -1,
        "random_state": config.RANDOM_STATE,
    }
    p = {**defaults, **(params or {})}
    return RandomForestRegressor(**p)



# MLP (PyTorch)


class _MLP(object):
    """Thin sklearn-compatible wrapper around a PyTorch MLP regressor."""

    def __init__(self, input_dim: int, hidden_dims: list[int],
                 dropout: float, epochs: int, batch_size: int, lr: float,
                 device: str = "cpu"):
        import torch
        import torch.nn as nn

        self.device = torch.device(device)
        self.epochs = epochs
        self.batch_size = batch_size

        layers: list[nn.Module] = []
        in_dim = input_dim
        for h in hidden_dims:
            layers += [nn.Linear(in_dim, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            in_dim = h
        layers.append(nn.Linear(in_dim, 1))
        self.net = nn.Sequential(*layers).to(self.device)

        self.optimizer = torch.optim.AdamW(self.net.parameters(), lr=lr, weight_decay=1e-4)
        self.scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            self.optimizer, T_max=epochs)
        self.criterion = nn.MSELoss()
        self.scaler = StandardScaler()

    def fit(self, X: np.ndarray, y: np.ndarray):
        import torch
        X = self.scaler.fit_transform(X)
        X_t = torch.tensor(X, dtype=torch.float32).to(self.device)
        y_t = torch.tensor(y, dtype=torch.float32).unsqueeze(1).to(self.device)

        dataset = torch.utils.data.TensorDataset(X_t, y_t)
        loader  = torch.utils.data.DataLoader(
            dataset, batch_size=self.batch_size, shuffle=True, drop_last=False)

        self.net.train()
        for epoch in range(1, self.epochs + 1):
            epoch_loss = 0.0
            for xb, yb in loader:
                self.optimizer.zero_grad()
                loss = self.criterion(self.net(xb), yb)
                loss.backward()
                self.optimizer.step()
                epoch_loss += loss.item()
            self.scheduler.step()
            if epoch % 10 == 0:
                log.info("    Epoch %3d/%d  loss=%.4f", epoch, self.epochs, epoch_loss / len(loader))
        return self

    def predict(self, X: np.ndarray) -> np.ndarray:
        import torch
        X = self.scaler.transform(X)
        X_t = torch.tensor(X, dtype=torch.float32).to(self.device)
        self.net.eval()
        with torch.no_grad():
            preds = self.net(X_t).squeeze(1).cpu().numpy()
        return preds

    def save(self, path: str):
        import torch
        torch.save(self.net.state_dict(), path)
        log.info("MLP weights → %s", path)


def build_mlp(input_dim: int, epochs: int | None = None) -> _MLP:
    try:
        import torch
        device = "cuda" if torch.cuda.is_available() else "cpu"
    except ImportError:
        log.error("torch not installed: pip install torch")
        sys.exit(1)

    log.info("MLP device: %s", device)
    return _MLP(
        input_dim=input_dim,
        hidden_dims=config.MLP_HIDDEN_DIMS,
        dropout=config.MLP_DROPOUT,
        epochs=epochs or config.MLP_EPOCHS,
        batch_size=config.MLP_BATCH_SIZE,
        lr=config.MLP_LR,
        device=device,
    )



# Optuna HPO  (optional, only if --tune is passed)
# ════════════════════════════════════════════════════════════════════════════

def tune_xgb(X_tr: np.ndarray, y_tr: np.ndarray, n_trials: int = 40) -> dict:
    try:
        import optuna
        optuna.logging.set_verbosity(optuna.logging.WARNING)
    except ImportError:
        log.warning("optuna not installed — skipping HPO. pip install optuna")
        return {}

    from xgboost import XGBRegressor
    from sklearn.model_selection import cross_val_score

    def objective(trial):
        params = {
            "n_estimators":      trial.suggest_int("n_estimators", 100, 600),
            "max_depth":         trial.suggest_int("max_depth", 3, 7),
            "learning_rate":     trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            "subsample":         trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "min_child_weight":  trial.suggest_int("min_child_weight", 1, 10),
            "reg_alpha":         trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
            "reg_lambda":        trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
            "random_state": config.RANDOM_STATE,
            "n_jobs": -1,
        }
        model = XGBRegressor(**params)
        scores = cross_val_score(model, X_tr, y_tr,
                                 cv=3, scoring="neg_root_mean_squared_error",
                                 n_jobs=1)
        return -scores.mean()

    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    log.info("Best HPO RMSE: %.4f", study.best_value)
    log.info("Best params:   %s", study.best_params)
    return study.best_params


# ═════════════════════════════════════════════════════════════════════════════
# Feature importance
# ═════════════════════════════════════════════════════════════════════════════

def save_feature_importance(model, feature_cols: list[str], model_name: str):
    """Save a feature importance CSV for tree-based models."""
    try:
        importances = model.feature_importances_
    except AttributeError:
        return  # MLP has no feature_importances_

    fi = pd.DataFrame({
        "feature": feature_cols,
        "importance": importances,
    }).sort_values("importance", ascending=False)

    path = os.path.join(config.RESULTS_DIR, f"{model_name}_feature_importance.csv")
    fi.to_csv(path, index=False)
    log.info("Feature importance → %s", path)
    log.info("Top 15 features:\n%s", fi.head(15).to_string(index=False))


# ═════════════════════════════════════════════════════════════════════════════
# Main
# ═════════════════════════════════════════════════════════════════════════════

def parse_args():
    p = argparse.ArgumentParser(description="Train viral-video regression model")
    p.add_argument("--model",  choices=["xgb", "lgbm", "mlp", "rf"],
                   default="xgb",
                   help="Model architecture (default: xgb)")
    p.add_argument("--no-cv",  action="store_true",
                   help="Skip cross-validation, train on full train split only")
    p.add_argument("--tune",   action="store_true",
                   help="Run Optuna HPO before final training (XGB only)")
    p.add_argument("--epochs", type=int, default=None,
                   help="Training epochs (MLP only, overrides config)")
    p.add_argument("--test-size", type=float, default=config.TEST_SIZE,
                   help="Fraction reserved for final hold-out test")
    return p.parse_args()


def main():
    args = parse_args()
    config.make_dirs()

    # ── Load data 
    X, y, feature_cols = load_dataset()

    # Train / test split 
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=args.test_size,
        random_state=config.RANDOM_STATE,
    )
    log.info("Train=%d  Test=%d", len(X_train), len(X_test))

    # Optional HPO 
    best_hpo_params: dict = {}
    if args.tune:
        if args.model != "xgb":
            log.warning("--tune is only supported for XGBoost; ignoring.")
        else:
            log.info("Running Optuna HPO (40 trials)…")
            best_hpo_params = tune_xgb(X_train, y_train, n_trials=40)

    # ── Cross-validation ───────────────────────────────────────────────────
    all_cv_results: dict = {}

    if not args.no_cv:
        log.info("▶ %d-fold cross-validation on training set…", config.CV_FOLDS)

        if args.model == "xgb":
            factory = lambda: build_xgb(best_hpo_params)
        elif args.model == "lgbm":
            factory = lambda: build_lgbm()
        elif args.model == "rf":
            factory = lambda: build_rf()
        else:
            # MLP — we skip CV to save time; it's done once below
            log.info("Skipping fold-CV for MLP (use --no-cv to suppress this notice).")
            factory = None

        if factory is not None:
            fold_metrics = cross_validate(factory, X_train, y_train)
            summary = summarise_cv(fold_metrics)
            all_cv_results["folds"] = fold_metrics
            all_cv_results["summary"] = summary
            log.info("CV summary: RMSE=%.4f±%.4f  MAE=%.4f±%.4f  R²=%.4f±%.4f",
                     summary["rmse"]["mean"], summary["rmse"]["std"],
                     summary["mae"]["mean"],  summary["mae"]["std"],
                     summary["r2"]["mean"],   summary["r2"]["std"])

    # Final model training 
    log.info("▶ Training final %s on full training set…", args.model.upper())
    t0 = time.time()

    if args.model == "xgb":
        model = build_xgb(best_hpo_params)
        model.fit(X_train, y_train,
                  eval_set=[(X_test, y_test)],
                  verbose=False)

    elif args.model == "lgbm":
        from lightgbm import early_stopping, log_evaluation
        model = build_lgbm()
        model.fit(X_train, y_train,
                  eval_set=[(X_test, y_test)],
                  callbacks=[early_stopping(50, verbose=False),
                              log_evaluation(period=-1)])

    elif args.model == "rf":
        model = build_rf()
        model.fit(X_train, y_train)

    elif args.model == "mlp":
        model = build_mlp(input_dim=X_train.shape[1], epochs=args.epochs)
        model.fit(X_train, y_train)

    elapsed = time.time() - t0
    log.info("Training done in %.1f s", elapsed)

    # Hold-out evaluation 
    log.info("▶ Evaluating on hold-out test set…")
    test_preds = model.predict(X_test)
    test_m = regression_metrics(y_test, test_preds)
    log.info("Test  RMSE=%.4f  MAE=%.4f  R²=%.4f",
             test_m["rmse"], test_m["mae"], test_m["r2"])

    all_cv_results["test"] = test_m
    all_cv_results["model"] = args.model
    all_cv_results["n_features"] = X.shape[1]
    all_cv_results["n_train"] = len(X_train)
    all_cv_results["n_test"]  = len(X_test)

    # Save results JSON 
    results_path = os.path.join(config.RESULTS_DIR, "cv_scores.json")
    with open(results_path, "w") as f:
        json.dump(all_cv_results, f, indent=2)
    log.info("Results → %s", results_path)

    # Save model 
    if args.model == "mlp":
        model_path = os.path.join(config.MODEL_DIR, "best_model_mlp.pt")
        model.save(model_path)
        # Also pickle the scaler so inference can use it
        scaler_path = os.path.join(config.MODEL_DIR, "mlp_scaler.pkl")
        with open(scaler_path, "wb") as f:
            pickle.dump(model.scaler, f)
        log.info("MLP scaler → %s", scaler_path)
    else:
        model_path = os.path.join(config.MODEL_DIR, "best_model.pkl")
        with open(model_path, "wb") as f:
            pickle.dump(model, f)
        log.info("Model → %s", model_path)

    # Feature importance (tree models)
    save_feature_importance(model, feature_cols, args.model)

    # Save metadata needed by inference
    meta = {
        "model_type":   args.model,
        "feature_cols": feature_cols,
        "target_col":   config.TARGET_COL,
        "pca_dir":      config.PCA_DIR,
    }
    meta_path = os.path.join(config.MODEL_DIR, "model_meta.json")
    with open(meta_path, "w") as f:
        json.dump(meta, f, indent=2)
    log.info("Model meta → %s", meta_path)

    print("\n✅ Training complete.")
    print(f"   Model type : {args.model.upper()}")
    print(f"   Test RMSE  : {test_m['rmse']:.4f}")
    print(f"   Test R²    : {test_m['r2']:.4f}")


if __name__ == "__main__":
    main()

```


## What `04_train_model.py` Produces

Expected outputs from the training step:

- `models/best_model.pkl` for XGB/LGBM/RF
- `models/best_model_mlp.pt` for MLP
- `models/model_meta.json`
- `results/cv_scores.json`
- `results/xgb_feature_importance.csv` or similar

The next cells only read those saved outputs.


In [ ]:
train_outputs = {
    "best model pickle": str(Path(config.MODEL_DIR) / "best_model.pkl"),
    "best MLP weights": str(Path(config.MODEL_DIR) / "best_model_mlp.pt"),
    "MLP scaler": str(Path(config.MODEL_DIR) / "mlp_scaler.pkl"),
    "model metadata": str(Path(config.MODEL_DIR) / "model_meta.json"),
    "CV scores": str(Path(config.RESULTS_DIR) / "cv_scores.json"),
    "XGB feature importance": str(Path(config.RESULTS_DIR) / "xgb_feature_importance.csv"),
    "LGBM feature importance": str(Path(config.RESULTS_DIR) / "lgbm_feature_importance.csv"),
    "RF feature importance": str(Path(config.RESULTS_DIR) / "rf_feature_importance.csv"),
}

pd.DataFrame([
    {"artifact": name, "path": path, "exists": exists(path)}
    for name, path in train_outputs.items()
])


## Model Metadata

In [ ]:
model_meta_path = Path(config.MODEL_DIR) / "model_meta.json"
model_meta = read_json_if_exists(model_meta_path)

if model_meta is None:
    print("No model_meta.json found:", model_meta_path)
else:
    print(json.dumps({k: v for k, v in model_meta.items() if k != "feature_cols"}, indent=2))
    print("Number of feature columns:", len(model_meta.get("feature_cols", [])))
    display(pd.DataFrame({"feature_col": model_meta.get("feature_cols", [])}).head(50))


## Cross-Validation and Test Results

In [ ]:
cv_scores_path = Path(config.RESULTS_DIR) / "cv_scores.json"
cv_scores = read_json_if_exists(cv_scores_path)

if cv_scores is None:
    print("No cv_scores.json found:", cv_scores_path)
else:
    print(json.dumps({k: v for k, v in cv_scores.items() if k != "folds"}, indent=2))

    if "folds" in cv_scores:
        folds_df = pd.DataFrame(cv_scores["folds"])
        display(folds_df)
        metric_cols = [c for c in ["rmse", "mae", "r2"] if c in folds_df.columns]
        if metric_cols:
            folds_df.set_index("fold")[metric_cols].plot(kind="bar", figsize=(9, 4))
            plt.title("Cross-Validation Metrics by Fold")
            plt.tight_layout()
            plt.show()


## Saved Feature Importance

In [ ]:
importance_candidates = [
    Path(config.RESULTS_DIR) / "xgb_feature_importance.csv",
    Path(config.RESULTS_DIR) / "lgbm_feature_importance.csv",
    Path(config.RESULTS_DIR) / "rf_feature_importance.csv",
    Path(config.RESULTS_DIR) / "feature_importance.csv",
]

fi = None
for path in importance_candidates:
    if path.exists():
        fi = pd.read_csv(path)
        print("Loaded:", path)
        break

if fi is None:
    print("No feature importance CSV found in", config.RESULTS_DIR)
else:
    display(fi.head(30))
    top_fi = fi.head(25).iloc[::-1]
    plt.figure(figsize=(8, 8))
    plt.barh(top_fi["feature"], top_fi["importance"], color="#2f5d7c")
    plt.xlabel("Importance")
    plt.title("Top Saved Feature Importances")
    plt.tight_layout()
    plt.show()


# Part 3: Prepare Data for Another Separate Model

This is the clean handoff point. It reads the parquet and feature-name list, then creates `X` and `y`. After this, you can add your own model cells below.


In [ ]:
if not df.empty:
    feature_names_path = Path(config.FEATURE_DIR) / "feature_names.json"
    feature_cols = read_json_if_exists(feature_names_path)
    if feature_cols is None:
        feature_cols = [c for c in df.columns if c not in config.LEAKY_COLS + ["video_id", "platform"]]

    feature_cols = [c for c in feature_cols if c in df.columns]
    X = df[feature_cols].copy()
    y = df[config.TARGET_COL].copy() if config.TARGET_COL in df.columns else None

    print("X shape:", X.shape)
    print("y shape:", None if y is None else y.shape)
    display(X.head())
else:
    X = pd.DataFrame()
    y = None
    print("No parquet loaded, so X/y were not created.")


## Add Your Own Visualization or Model Cells Below

Examples you may add:

```python
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=config.RANDOM_STATE)
model = Ridge()
model.fit(X_train, y_train)
preds = model.predict(X_test)
print(mean_squared_error(y_test, preds) ** 0.5, r2_score(y_test, preds))
```

This should be the place where your new experiments start.
